# 10. Mixture-of-Experts — DeepSeekMoE, K3 Stable LatentMoE, and V4 MTP

This notebook implements the actual routing/data paths with small dimensions:

- SwiGLU and DeepSeekMoE fine-grained/shared-expert structure
- Kimi K3 SiTU-GLU
- sigmoid routing where balancing bias affects dispatch but not mixture weights
- Stable LatentMoE: down-project -> routed latent experts -> RMSNorm -> up-project
- full-width shared experts
- a small Quantile-Balancing bias update
- DeepSeek-V4/V3 lineage Multi-Token Prediction (MTP) training heads


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


## 1. SwiGLU expert


In [ ]:
class SwiGLUExpert(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim=None):
        super().__init__()
        output_dim = input_dim if output_dim is None else output_dim

        self.gate = nn.Linear(input_dim, hidden_dim, bias=False)
        self.value = nn.Linear(input_dim, hidden_dim, bias=False)
        self.out = nn.Linear(hidden_dim, output_dim, bias=False)

    def forward(self, x):
        return self.out(
            F.silu(self.gate(x)) * self.value(x)
        )


## 2. DeepSeekMoE: fine-grained routed experts + shared experts


In [ ]:
class TinyDeepSeekMoE(nn.Module):
    def __init__(
        self,
        model_dim=24,
        coarse_experts=4,
        coarse_top_k=2,
        segmentation=2,
        shared_experts=1,
        coarse_hidden=32,
    ):
        super().__init__()

        total_fine = coarse_experts * segmentation
        active_fine = coarse_top_k * segmentation
        expert_hidden = coarse_hidden // segmentation

        self.routed_count = total_fine - shared_experts
        self.routed_top_k = active_fine - shared_experts

        self.shared = nn.ModuleList(
            [
                SwiGLUExpert(model_dim, expert_hidden)
                for _ in range(shared_experts)
            ]
        )
        self.routed = nn.ModuleList(
            [
                SwiGLUExpert(model_dim, expert_hidden)
                for _ in range(self.routed_count)
            ]
        )
        self.router = nn.Linear(model_dim, self.routed_count, bias=False)

    def forward(self, tokens):
        shared_output = sum(
            expert(tokens) for expert in self.shared
        )

        scores = self.router(tokens).softmax(dim=-1)
        weights, ids = scores.topk(self.routed_top_k, dim=-1)

        routed_output = torch.zeros_like(tokens)

        for slot in range(self.routed_top_k):
            slot_ids = ids[:, slot]
            slot_weights = weights[:, slot]

            for expert_id, expert in enumerate(self.routed):
                mask = slot_ids == expert_id
                if mask.any():
                    routed_output[mask] += (
                        slot_weights[mask, None]
                        * expert(tokens[mask])
                    )

        return tokens + shared_output + routed_output, ids


deepseek_moe = TinyDeepSeekMoE().to(device)
tokens = torch.randn(12, 24, device=device)
deepseek_output, deepseek_ids = deepseek_moe(tokens)

print("DeepSeekMoE:", deepseek_output.shape)
print("routed ids:", deepseek_ids[:4])


## 3. SiTU-GLU

K3 uses bounded tanh factors around the gate/value projections.
This keeps the important activation control rather than replacing it by SwiGLU.


In [ ]:
class SiTUGLU(nn.Module):
    def __init__(
        self,
        input_dim,
        hidden_dim,
        output_dim,
        beta_gate=4.0,
        beta_value=25.0,
    ):
        super().__init__()
        self.beta_gate = beta_gate
        self.beta_value = beta_value

        self.gate = nn.Linear(input_dim, hidden_dim, bias=False)
        self.value = nn.Linear(input_dim, hidden_dim, bias=False)
        self.out = nn.Linear(hidden_dim, output_dim, bias=False)

    def forward(self, x):
        gate_pre = self.gate(x)
        value_pre = self.value(x)

        gate = (
            self.beta_gate
            * torch.tanh(gate_pre / self.beta_gate)
            * torch.sigmoid(gate_pre)
        )
        value = (
            self.beta_value
            * torch.tanh(value_pre / self.beta_value)
        )
        return self.out(gate * value)


## 4. Quantile-balancing dispatch

Router probability is `sigmoid(W_r x)`.
A balancing bias changes **which experts are dispatched**.
The actual mixture weight is normalized from the original sigmoid scores,
so the bias does not directly distort expert output magnitude.

The update below uses the top-k cutoff margin to move overloaded and
underloaded experts toward a more balanced allocation.


In [ ]:
@torch.no_grad()
def quantile_balance_update(
    router_scores,
    routing_bias,
    top_k,
    learning_rate=0.05,
):
    biased_scores = router_scores + routing_bias
    sorted_scores, _ = biased_scores.sort(dim=-1, descending=True)

    cutoff = sorted_scores[:, top_k - 1]
    next_cutoff = sorted_scores[:, top_k]
    target_quantile = 0.5 * (cutoff + next_cutoff)

    selected = biased_scores.topk(top_k, dim=-1).indices
    expert_count = router_scores.size(-1)

    load = torch.stack(
        [(selected == expert_id).float().mean()
         for expert_id in range(expert_count)]
    )
    target_load = top_k / expert_count

    routing_bias.add_(
        learning_rate * (target_load - load)
    )

    return {
        "load": load,
        "cutoff_mean": target_quantile.mean(),
    }


## 5. Stable LatentMoE

Routed experts work in a smaller latent width:

`x -> W_down -> routed SiTU experts -> weighted sum -> RMSNorm -> W_up`

Shared experts stay on the full model width and are always active.


In [ ]:
class TinyStableLatentMoE(nn.Module):
    def __init__(
        self,
        model_dim=24,
        latent_dim=12,
        routed_experts=8,
        top_k=2,
        shared_experts=1,
        latent_hidden=24,
    ):
        super().__init__()
        self.top_k = top_k
        self.routed_count = routed_experts

        self.down = nn.Linear(model_dim, latent_dim, bias=False)
        self.latent_norm = nn.RMSNorm(latent_dim)
        self.up = nn.Linear(latent_dim, model_dim, bias=False)

        self.router = nn.Linear(model_dim, routed_experts, bias=False)
        self.routing_bias = nn.Parameter(
            torch.zeros(routed_experts),
            requires_grad=False,
        )

        self.routed = nn.ModuleList(
            [
                SiTUGLU(
                    latent_dim,
                    latent_hidden,
                    latent_dim,
                )
                for _ in range(routed_experts)
            ]
        )
        self.shared = nn.ModuleList(
            [
                SiTUGLU(
                    model_dim,
                    2 * model_dim,
                    model_dim,
                )
                for _ in range(shared_experts)
            ]
        )

    def forward(self, tokens):
        latent = self.down(tokens)

        raw_scores = torch.sigmoid(self.router(tokens))
        dispatch_scores = raw_scores + self.routing_bias

        _, ids = dispatch_scores.topk(self.top_k, dim=-1)
        chosen_raw = raw_scores.gather(-1, ids)
        weights = chosen_raw / chosen_raw.sum(
            dim=-1,
            keepdim=True,
        ).clamp_min(1e-8)

        routed_output = torch.zeros_like(latent)

        for slot in range(self.top_k):
            slot_ids = ids[:, slot]
            slot_weights = weights[:, slot]

            for expert_id, expert in enumerate(self.routed):
                mask = slot_ids == expert_id
                if mask.any():
                    routed_output[mask] += (
                        slot_weights[mask, None]
                        * expert(latent[mask])
                    )

        routed_full = self.up(
            self.latent_norm(routed_output)
        )
        shared_full = sum(
            expert(tokens) for expert in self.shared
        )

        return tokens + routed_full + shared_full, {
            "raw_scores": raw_scores,
            "ids": ids,
            "weights": weights,
        }


k3_moe = TinyStableLatentMoE().to(device)
k3_tokens = torch.randn(16, 24, device=device)
k3_output, k3_diag = k3_moe(k3_tokens)

balance = quantile_balance_update(
    k3_diag["raw_scores"].detach(),
    k3_moe.routing_bias,
    k3_moe.top_k,
)

loss = k3_output.square().mean()
loss.backward()

print("Stable LatentMoE:", k3_output.shape)
print("selected experts:", k3_diag["ids"][:4])
print("normalized mixture sums:",
      k3_diag["weights"].sum(dim=-1)[:4])
print("expert load:", balance["load"])
print("latent down-proj grad:", k3_moe.down.weight.grad.norm().item())


## 6. Multi-Token Prediction (MTP)

DeepSeek-V4 retains the V3 MTP strategy.
A small language model therefore predicts several future offsets during training
rather than silently dropping this training-time path.


In [ ]:
class TinyMTPModel(nn.Module):
    def __init__(
        self,
        vocab=64,
        model_dim=24,
        depth=2,
        future_offsets=(1, 2),
    ):
        super().__init__()
        self.future_offsets = future_offsets
        self.embedding = nn.Embedding(vocab, model_dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=model_dim,
            nhead=3,
            dim_feedforward=4 * model_dim,
            batch_first=True,
        )
        self.backbone = nn.TransformerEncoder(
            encoder_layer,
            num_layers=depth,
        )
        self.norm = nn.RMSNorm(model_dim)
        self.heads = nn.ModuleList(
            [
                nn.Linear(model_dim, vocab, bias=False)
                for _ in future_offsets
            ]
        )

    def forward(self, token_ids):
        length = token_ids.size(1)
        causal_mask = torch.triu(
            torch.ones(
                length,
                length,
                dtype=torch.bool,
                device=token_ids.device,
            ),
            diagonal=1,
        )

        hidden = self.backbone(
            self.embedding(token_ids),
            mask=causal_mask,
        )
        hidden = self.norm(hidden)
        return [head(hidden) for head in self.heads]


mtp = TinyMTPModel().to(device)
token_ids = torch.randint(0, 64, (3, 10), device=device)
mtp_logits = mtp(token_ids)

mtp_loss = torch.zeros((), device=device)
for offset, logits in zip(mtp.future_offsets, mtp_logits):
    valid_logits = logits[:, :-offset]
    targets = token_ids[:, offset:]
    mtp_loss = mtp_loss + F.cross_entropy(
        valid_logits.reshape(-1, valid_logits.size(-1)),
        targets.reshape(-1),
    )

mtp_loss.backward()
print("MTP loss:", mtp_loss.item())
print("future heads:", len(mtp_logits))


## References and provenance

- **DeepSeekMoE**: fine-grained expert segmentation and shared-expert isolation.
- **Kimi K3 Stable LatentMoE**: highly sparse routed experts in latent width,
  full-width shared experts, SiTU-GLU, sigmoid routing, and Quantile Balancing.
- **DeepSeek-V4**: official model card states that V4 retains the
  Multi-Token Prediction strategy from V3.
